# LLM 서비스 개발 과정 — Day 3 (2026-03-26, 목)

## 프롬프트 엔지니어링 (3) + 평가(Evaluation) + CoT

> **모두의연구소 재직자 LLM 6기 · 3주차 Day 3**
> *Few-shot 예시 선택 전략 · 프롬프트 성능 평가 · Chain-of-Thought 추론*

---

### 오늘 배울 것 한눈에

| 파트 | 주제 | 핵심 키워드 |
|------|------|-------------|
| **1** | Few-shot 예시 고르기 원칙 | 다양성 · 관련성 · 균형 |
| **2** | FewShotPromptTemplate / FewShotChatMessagePromptTemplate | 정적 예시 주입 |
| **3** | SemanticSimilarityExampleSelector | **동적**으로 예시 선택 (의미 기반) |
| **4** | Zero-shot vs Few-shot **정량 평가** | `eval_dataset`, accuracy |
| **5** | 평가 메트릭 개념 정리 | classification / regression / BLEU · ROUGE |
| **6** | LLM이 **잘 틀리는 문제** 체험 | 말 경주, 몬티홀, Sally 형제, LOLLAPALOOZA |
| **7** | **Chain-of-Thought (CoT)** | "단계별로 생각해 보세요" 한 줄의 힘 |
| **8** | Few-shot CoT | 예시 + 추론 과정 동시 제공 |

### 오늘의 한 줄 비유
> Few-shot이 **"시험 전 족보 몇 장을 보여주는 것"** 이라면,
> CoT는 **"답만 말하지 말고 풀이 과정을 써라"** 고 강제하는 것이에요.
> 둘 다 모델을 "더 똑똑하게" 만드는 게 아니라, **더 신중하게 답하도록 유도**하는 장치입니다.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w3_memory_prompt_engineering/llm_260326_prompt_engineering_and_evaluation.ipynb)


---
## Step 0–3. Colab 환경 설정 (매 세션 1회)

Colab Secrets에 `OPENAI_API_KEY`를 등록한 뒤 아래 셀들을 순서대로 실행하세요.
(이미 설치/세팅돼 있어도 중복 실행에 안전합니다.)


In [ ]:
!pip install -q faiss-cpu langchain langchain-community langchain-core langchain-openai openai pandas

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import FewShotChatMessagePromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.messages import AIMessage
from langchain_core.messages import SystemMessage

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

---
## Part 1. Few-shot 예시, **어떻게 고를까**

어제까지 Few-shot 프롬프팅을 해봤어요. 그런데 강의에서 선생님이 강조한 한 가지:
> **"예시 선택이 Few-shot 성능의 70% 이상을 결정한다"** (관련 논문 결과)

어떤 예시를 넣어야 할까? 선생님이 제시한 **3가지 원칙**입니다.

### 1) 다양성 (Diversity)
다양한 케이스를 커버하는 예시를 넣어 **편향(bias)** 을 방지.
(예: 리뷰 감정 분류 → 긍정만 넣지 말고 부정·중립·혼합도 섞어야 함)

### 2) 관련성 (Relevance)
풀려는 태스크와 **같은 도메인**의 예시를 넣기.
(예: 감정 분류인데 뉴스 요약 예시를 넣지는 않죠)

### 3) 균형 (Balance)
각 **클래스(라벨)가 고르게 분포**되게. AI는 majority-label bias에 약하거든요.
긍정 8개 + 부정 2개 넣으면 → 모델이 "긍정"이라고 답하기 쉬움.

### 비유
Few-shot 예시는 **"시험 직전 족보"** 같은 거예요.
- **다양성** = 족보에 여러 유형 문제가 있어야 실전에서 당황 안 함
- **관련성** = 수학 시험 보는데 영어 족보를 주면 안 되죠
- **균형** = 객관식 답이 ①만 가득하면, 학생이 "답은 아마 ①" 하고 찍는 습관이 생김

### 참고 논문들
- Order sensitivity of few-shot prompts: https://arxiv.org/pdf/2102.09690
- What makes good in-context examples (kNN 기반 선택): https://arxiv.org/pdf/2101.06804
- Rethinking the role of demonstrations (라벨 랜덤화 실험): https://arxiv.org/pdf/2202.12837
- 다양성 극대화 샘플 선택: https://arxiv.org/pdf/2209.01975


In [ ]:
# 다양성 / 관련성 / 균형 — 오늘의 키워드 3개
# 참고 논문 URL은 위 마크다운 셀에 정리되어 있어요.


---
## Part 2. LangChain Few-shot 템플릿 2종

### FewShotPromptTemplate (구형·문자열 기반)
`prefix + 예시들(example_separator로 구분) + suffix` 구조로 **하나의 긴 문자열**을 만들어 줘요.

### FewShotChatMessagePromptTemplate (신형·채팅 메시지 기반)
예시 하나를 `("human", 입력) + ("ai", 출력)` 한 쌍의 메시지로 변환.
GPT-4o-mini 같은 **채팅 모델에는 이게 궁합이 좋습니다.**

### 비유
- FewShotPromptTemplate = "예시들을 하나의 긴 편지로 이어 붙여서 보내기"
- FewShotChatMessagePromptTemplate = "예시마다 '이럴 땐 이렇게 답해요' 하고 카톡 메시지 주고받은 척 위장하기"


In [ ]:
# Few-shot 관련 프롬프트 클래스 4종 임포트
from langchain_core.prompts import (
    FewShotPromptTemplate,             # 문자열 기반 (구형)
    PromptTemplate,                    # 단일 문자열 템플릿
    ChatPromptTemplate,                # 채팅 메시지 템플릿
    FewShotChatMessagePromptTemplate,  # 채팅 메시지 기반 Few-shot (신형)
)


In [ ]:
# 감정 분류용 예시 4개 — 긍정/부정/중립/혼합 4개 클래스를 **균형 있게** 포함
examples = [
    {"input": "이 제품 정말 최고입니다!", "output": "긍정"},
    {"input": "품질이 너무 안좋아요", "output": "부정"},
    {"input": "보통이에요, 무난합니다.", "output": "중립"},
    {"input": "디자인은 좋은데 성능이 아쉬워요", "output": "혼합"},
]


In [ ]:
# ─── FewShotPromptTemplate (문자열 기반) ───
# 각 예시 하나를 어떤 포맷으로 렌더링할지 정의
example_template = PromptTemplate(
    input_variables=["input", "output"],
    template="리뷰: {input}\n감정: {output}",  # 예: "리뷰: 최고입니다!\n감정: 긍정"
)

# 최종 프롬프트 = prefix + 예시들 + suffix
fewshot_prompt = FewShotPromptTemplate(
    examples=examples,                   # 위에서 만든 예시 리스트
    example_prompt=example_template,     # 예시 하나 렌더링 규칙
    prefix="다음 예시를 참고해서 리뷰의 감정을 분류해주세요.\n",  # 맨 앞 설명
    suffix="\n리뷰: {input}\n감정:",     # 실제 질문 자리 (감정: 뒤에 모델이 답함)
    input_variables=["input"],           # suffix에서 쓰이는 변수
    example_separator="\n\n",           # 예시 사이 구분자 (한 줄 띄움)
)


In [ ]:
# 템플릿이 **실제로 어떤 문자열**을 만드는지 확인
formatted = fewshot_prompt.format(input="가격은 비싸지만 품질이 뛰어나요")
print(formatted)
# → prefix + 예시 4개 + suffix(리뷰: 가격은..., 감정:) 형태로 조립됨


In [ ]:
# ─── FewShotChatMessagePromptTemplate (채팅 메시지 기반) ───
# 예시 하나를 ("human", 입력) + ("ai", 출력) 메시지 쌍으로 변환
example_prompt_chat = ChatPromptTemplate.from_messages([
    ("human", "리뷰 : {input}"),
    ("ai", "감정: {output}"),
])

# 이 클래스가 examples를 돌며 여러 메시지 쌍을 쏟아냄
fewshot_chat_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt_chat,
)


In [ ]:
# 최종 채팅 프롬프트 = 시스템 메시지 + Few-shot 메시지들 + 실제 질문
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 감정 분석 전문가입니다, 리뷰의 감정을 긍정/부정/중립/혼합 중 하나로 분류해주세요"),
    fewshot_chat_prompt,                 # 위에서 만든 Few-shot 메시지 묶음이 여기 펼쳐짐
    ("human", "리뷰 : {input}"),
])


In [ ]:
# LCEL: prompt | llm — 파이프 하나로 체인 완성
chain = final_prompt | llm
result = chain.invoke({"input": "가격은 비싸지만 품질이 뛰어나요"})
print(result.content)
# → Few-shot 예시 덕분에 "혼합" 같은 라벨로 정확히 분류되는 걸 기대


---
## Part 3. 예시가 **너무 많을 때** — Semantic Similarity Selector

예시 풀이 10개 정도면 그냥 다 넣어도 되지만, **100개·1000개**라면?
- 토큰 비용 폭발
- 모델 집중력 분산 (lost-in-the-middle)

그래서 **쿼리와 의미적으로 비슷한 k개만 골라서** 프롬프트에 넣는 기법이 `SemanticSimilarityExampleSelector`.

### 동작 원리 (2단계)
1. 모든 예시 → **임베딩 벡터**로 변환 → `InMemoryVectorStore`에 저장
2. 쿼리가 들어오면 쿼리도 임베딩 → 저장된 벡터 중 **코사인 유사도 top-k** 선택

### 비유
사서가 100권 중에서 **"지금 내 질문과 가장 비슷한 주제의 책 3권"** 만 뽑아 주는 거예요.
전체 장서를 다 훑지 않으니 빠르고, 질문에 맞춤한 예시만 들어가니 LLM도 집중하기 쉬워요.


In [ ]:
# 큰 예시 풀 (10개) — 실제론 100개, 1000개짜리일 수도
large_examples = [
    {"input": "정말 만족합니다. 강력 추천!", "output": "긍정"},
    {"input": "배송이 빠르고 포장이 꼼꼼해요.", "output": "긍정"},
    {"input": "가격 대비 훌륭합니다.", "output": "긍정"},
    {"input": "품질이 최악입니다. 환불 요청했어요.", "output": "부정"},
    {"input": "고객센터 응대가 너무 불친절합니다.", "output": "부정"},
    {"input": "택배가 분실되었어요.", "output": "부정"},
    {"input": "보통이에요. 특별한 점은 없습니다.", "output": "중립"},
    {"input": "사진과 동일한 제품입니다.", "output": "중립"},
    {"input": "디자인은 예쁜데 내구성이 약해요.", "output": "혼합"},
    {"input": "기능은 많은데 사용법이 복잡합니다.", "output": "혼합"},
]


In [ ]:
# Semantic = "뜻이 비슷한", Similarity = "유사도" → 의미 기반 예시 셀렉터
from langchain_core.example_selectors import SemanticSimilarityExampleSelector


In [ ]:
# 메모리 기반 벡터 스토어 (FAISS 안 쓰고도 간단 검색 가능)
# 실습용·소규모에 최적, 노트북 세션 끝나면 사라짐
from langchain_core.vectorstores import InMemoryVectorStore


In [ ]:
# 예시 셀렉터 초기화: 예시들을 임베딩해서 InMemoryVectorStore에 저장, k=3개 뽑기
example_selector = SemanticSimilarityExampleSelector.from_examples(
    large_examples,                                    # 예시 풀
    OpenAIEmbeddings(model="text-embedding-3-small"),  # 임베딩 모델
    InMemoryVectorStore,                               # 벡터 저장소 클래스
    k=3,                                                # top-3 선택
)


In [ ]:
# 쿼리 3개를 돌려보며, 각 쿼리에 대해 어떤 예시가 뽑히는지 확인
queries = [
    "배송이 너무 느려요",
    "값은 좀 나가지만 성능은 훌륭해요",
    "무난한 제품이에요",
]

for q in queries:
    selected = example_selector.select_examples({"input": q})
    print(f"query : {q}")
    for s in selected:
        print(f" selected : {s['output']}, {s['input']}")
    print()
# → "배송이 느려요"면 배송/고객센터 관련 예시가, "성능 훌륭해요"면 가격/성능 예시가 뽑힘


In [ ]:
# 동적 셀렉터를 FewShotChatMessagePromptTemplate에 연결
# → 쿼리마다 **다른** 예시 3개가 프롬프트에 주입됨
dynamic_few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,    # 고정 examples= 대신 selector 사용
    example_prompt=example_prompt_chat,   # 예시 하나의 렌더링 포맷 (앞에서 정의)
)

# 최종 프롬프트 조립
dynamic_final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 감정 분석 전문가입니다. 리뷰의 감정을 긍정/부정/중립/혼합 중 하나로 분류하세요"),
    dynamic_few_shot_prompt,
    ("human", "리뷰 : {input}"),
])

dynamic_chain = dynamic_final_prompt | llm


In [ ]:
# 동적 Few-shot 체인 실행 — 쿼리마다 예시가 달라지는 걸 확인
test_inputs = ["택배 상자가 찌그러져왔어요", "화면은 선명한데 스피커 소리가 작아요"]
for t in test_inputs:
    result = dynamic_chain.invoke({"input": t})
    print(f"{t} : {result.content}")


---
## Part 4. 프롬프트 **성능 평가** — 숫자로 말하기

> "이 프롬프트 괜찮은 것 같아요" → 팀장: **"근거가 뭔데요?"**

프롬프트를 바꿀 때마다 **감**으로 평가하면 설득력이 없어요. ML/AI 팀의 기본 문법은:
1. **평가 데이터셋(eval_dataset)** 을 만든다 (정답 포함)
2. 각 프롬프트 버전으로 예측을 돌린다
3. **정답과 비교** → accuracy, precision, recall 등 지표로 변환

### 비유
프롬프트는 **학생**, `eval_dataset`은 **시험지**예요.
> Zero-shot 학생 vs Few-shot 학생, 누가 시험지에서 몇 점 맞는지 공정하게 비교.

### 오늘 할 것
- `evaluate_zero_shot()`: 시스템 메시지만 주고 맞히게 함
- `evaluate_few_shot()`: 예시 몇 개 보여준 뒤 맞히게 함
- 결과를 `pd.DataFrame`으로 정리 → 표 비교


In [ ]:
# 평가용 데이터셋 — 8개 샘플 (정답 포함)
# 실전에선 100~1000개 규모로, 난이도도 섞어서 만들어요
eval_dataset = [
    {"text": "완벽한 제품입니다! 강력 추천합니다.", "expected": "긍정"},
    {"text": "두 번 다시 구매하지 않겠습니다.", "expected": "부정"},
    {"text": "평범합니다. 특별히 좋지도 나쁘지도 않아요.", "expected": "중립"},
    {"text": "기능은 좋은데 AS가 아쉬워요.", "expected": "혼합"},
    {"text": "가격도 착하고 품질도 좋아요.", "expected": "긍정"},
    {"text": "사진과 너무 달라서 실망했습니다.", "expected": "부정"},
    {"text": "그냥 쓸만 합니다.", "expected": "중립"},
    {"text": "배송은 빠른데 제품이 기대 이하에요.", "expected": "혼합"},
]


In [ ]:
# Zero-shot 평가 함수
# 예시 없이 "이 라벨 중 하나로만 답해" 시스템 메시지만 주고 정확도 체크
def evaluate_zero_shot(dataset):
    results = []
    for item in dataset:
        response = llm.invoke([
            SystemMessage(content="리뷰의 감정을 분류하세요. 반드시 '긍정', '부정', '중립', '혼합' 중 하나만 출력하세요"),
            HumanMessage(content=item["text"]),
        ]).content

        predicted = response.strip()  # 앞뒤 공백/개행 제거
        results.append({
            "text": item["text"],
            "expected": item["expected"],
            "predicted": predicted,
            "correct": predicted == item["expected"],  # True/False
        })
    return results


In [ ]:
# Few-shot 평가 함수
# SystemMessage + (HumanMessage/AIMessage) 쌍으로 예시를 모델 대화 히스토리처럼 주입
def evaluate_few_shot(dataset, examples):
    results = []
    base_messages = [SystemMessage(content="리뷰의 감정을 분류하세요. 반드시 '긍정', '부정', '중립', '혼합' 중 하나만 출력하세요")]

    # 예시들을 human/ai 메시지 쌍으로 넣어줌 (모델이 대화 히스토리처럼 학습)
    for ex in examples:
        base_messages.append(HumanMessage(content=ex["input"]))
        base_messages.append(AIMessage(content=ex["output"]))

    for item in dataset:
        messages = base_messages + [HumanMessage(content=item["text"])]
        response = llm.invoke(messages).content

        predicted = response.strip()
        results.append({
            "text": item["text"],
            "expected": item["expected"],
            "predicted": predicted,
            "correct": predicted == item["expected"],
        })
    return results


In [ ]:
# Few-shot에 쓸 예시 — **eval_dataset과 겹치면 컨닝**이니까 절대 안 됨!
# (모델이 시험 답안을 그대로 외운 걸 테스트하는 꼴)
few_shot_examples = [
    {"input": "정말 만족합니다. 강력 추천!", "output": "긍정"},
    {"input": "배송이 빠르고 포장이 꼼꼼해요.", "output": "긍정"},
    {"input": "가격 대비 훌륭합니다.", "output": "긍정"},
    {"input": "품질이 최악입니다. 환불 요청했어요.", "output": "부정"},
]

# 두 방식 각각 평가
zero_results = evaluate_zero_shot(eval_dataset)
fewshot_results = evaluate_few_shot(eval_dataset, few_shot_examples)


In [ ]:
# raw 결과 확인 (list of dict)
zero_results


In [ ]:
fewshot_results


In [ ]:
# pandas로 예쁘게 표 출력
import pandas as pd


In [ ]:
# Few-shot 결과를 DataFrame으로 — 엑셀 표처럼 보임
pd.DataFrame(fewshot_results)  # .to_csv('a.csv') 하면 CSV로도 저장 가능


In [ ]:
# Zero-shot 결과도 표로
pd.DataFrame(zero_results)


---
## Part 5. 평가 메트릭, 어떤 게 있나

지금은 단순 `예측 == 정답` (accuracy)만 봤지만, 태스크 유형별로 쓰는 지표가 달라요.

### Classification (분류 태스크)
- **Accuracy**: 맞춘 비율
- **Precision / Recall / F1**: 클래스 불균형일 때 필수
- **Confusion Matrix**: 어느 클래스를 어느 클래스와 헷갈리는지

### Regression (회귀 태스크 — 숫자 예측)
긍정 1.0 ~ 부정 0.0 같은 **점수**로 예측할 때.
- **MSE (Mean Squared Error)**: 오차 제곱 평균
- **RMSE**: MSE의 제곱근 (같은 단위로 해석 가능)
- **MAE**: 절댓값 오차 평균

### NLP Generation (번역 / 요약 / 문구 생성)
- **BLEU**: 정답과 모델 출력의 **n-gram 겹침** 비율
  예) 정답 "I go to school" / 모델 "I go school" → BLEU ≈ 0.75
- **ROUGE**: 주로 요약 평가에 사용 (recall 기반)

### 비유
- Classification = **객관식 채점** (O/X)
- Regression = **숫자 맞히기** (가까울수록 좋음)
- BLEU/ROUGE = **서술형 답안지 채점** (겹치는 단어 세기)


In [ ]:
# classification 관련 metric
# 레이블이 아니라 점수로 평가한다면? 긍정(1.0) ~ 부정(0.0)
# 예) 정답 1.0  예측 0.9 → 오차 = sqrt((1-0.9)**2) = 0.1
#     정답 0.0  예측 0.2 → 오차 = 0.2


In [ ]:
# regression 관련 metric: RMSE (Root Mean Squared Error), MAE, MSE ...
# 필요할 때 "classification metrics" / "regression metrics" 검색어로 찾으면 됨


In [ ]:
# 번역/요약/문구 생성: BLEU, ROUGE
# 예) 정답: "I go to school"  모델: "I go school"  → BLEU ≈ 0.75 (3/4 단어 겹침)


---
## Part 6. LLM이 **잘 틀리는 문제들** — 왜?

프롬프트 엔지니어링 후반부로 가기 전, **LLM이 왜 약한지**를 먼저 체감해 봅시다.

### 공통 패턴
- **추론이 필요한** 문제 (한 번에 답 안 나옴)
- **유명한 문제의 변형** → 원본 답을 그대로 뱉어 버림 (패턴 복사)
- **토큰 단위 정밀 계산** (글자 수 세기, 철자 단위)

### 오늘 볼 문제들
1. **25마리 말 경주** — 3마리 빠른 말 찾기 (답: 7번)
2. **6마리 말** 변형 — 한 번에 다 뛸 수 있음 (답: 1번!)
3. **Sally의 여자 형제** — 관계 추론
4. **LOLLAPALOOZA의 L 개수** — 철자 세기
5. **STRAWBERRY의 R 개수** — 유명한 실패 사례
6. **몬티홀 변형** — 사회자가 문을 안 열어준 경우 (답: 확률 1/3로 똑같음)

### 비유
LLM은 **"수능 족보 암기는 잘하는데, 문제를 살짝 비틀면 멘붕하는 학생"** 이에요.
6마리 말 문제는 "25마리 말 경주 문제"의 변형인데, 원본 답(7번)을 갖다 붙이는 실수를 잘 해요.

### 재미있는 사이트
[llm-quiz.com](https://www.llm-quiz.com/quiz) — 이런 함정 문제 모음. 심심할 때 돌려보세요.


In [ ]:
# 원본 문제 (유명한 추론 문제)
# 말이 25마리, 한 번에 5마리만 경주 가능
# 가장 빠른 3마리 말을 찾으려면 몇 번의 경주?
# (정답: 7번)
result = llm.invoke("말이 25마리 있습니다. 한 번에 5마리만 경주를 할 수 있습니다. 가장 빠른 3마리 말을 찾으려면 몇 번의 경주를 해야할까요?")
print(result.content)


In [ ]:
# **변형** 문제: 6마리 말 + 한 번에 6마리 다 뛸 수 있음 → 답은 당연히 **1번**
# 근데 LLM은 원본(25마리 문제)을 기억해서 여러 번이라고 답하기 쉬움
result = llm.invoke("말이 6마리가 있고, 가장 빠른 말을 찾고 싶습니다. 한 번에 최대 6마리가 동시에 뛸 수 있습니다. 최소 몇 번의 경주가 필요할까요?")
print(result.content)


In [ ]:
# 몬티홀 **변형** — 원본은 "사회자가 다른 문을 열어 야채 공개 후 바꾸시겠냐"
# 여기서는 "**아직 문을 열지 않은 상태**에서 바꾸시겠냐" → 확률 1/3 동일, 바꿔도 똑같음
# 하지만 LLM은 원본 몬티홀 답(바꾸는 게 유리, 2/3)을 패턴 복사해서 틀림
llm.invoke("3개 문 중 하나에 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다. 사회자는 아직 어떤 문도 열지 않았습니다. 바꾸는 것이 유리할까요?")


In [ ]:
# Sally의 여자 형제 문제 (관계 추론)
# Sally(여자) 남자 형제 3명 → 각 남자 형제의 여자 형제 2명 = Sally + 1명
# 따라서 Sally의 여자 형제는 1명
print(llm.invoke("""Sally(여자)한테 남자 형제가 3명 있어요.
각 남자 형제한테는 여자 형제가 2명 있어요.
Sally의 여자 형제는 몇 명일까요?""").content)


In [ ]:
# 철자 세기 — LLM은 **토큰 단위**로 보기 때문에 글자 개수 세는 것도 자주 틀림
print(llm.invoke("LOLLAPALOOZA 라는 단어에서 L이 몇 번 나타나나요?").content)
# (정답: 4번)


---
## Part 7. **Chain-of-Thought (CoT)** — "단계별로 생각해 보세요"

### 논문 한 장 요약
[*Large Language Models are Zero-Shot Reasoners*](https://arxiv.org/abs/2205.11916)
> **프롬프트 끝에 "단계별로 생각해 보세요" 한 줄만 추가해도 추론 정확도가 극적으로 올라간다**

### 왜 효과가 있나?
LLM은 **생성한 토큰을 입력 컨텍스트로 다시 활용**하는 방식이라, 중간 단계를 글로 쓰게 하면 스스로 검산하는 효과가 생겨요. (= "속으로 계산하지 말고 종이에 풀이 적어!")

### CoT의 종류
1. **Zero-shot CoT**: 그냥 "단계별로 생각해 보세요" 붙이기
2. **Few-shot CoT**: 예시에 **풀이 과정**까지 적어서 보여주기
3. **조건 나열 CoT**: "문제의 조건을 전부 나열한 뒤 각각 검토" 식
4. **변형 체크 CoT**: "이 문제가 기존 유명 문제와 **어떻게 다른지 분석**한 뒤 답해줘"

### 비유
CoT는 학생한테 **"암산하지 말고 풀이과정 써!"** 라고 시키는 거예요.
중간 과정을 쓰게 하면 무리수 연산/조건 누락이 줄어들죠. LLM도 똑같아요.


In [ ]:
# CoT 패턴
# 1. CoT (Chain-of-Thought): "단계별로 생각해 보세요" 한 줄 추가
# 2. few-shot CoT: 예시 + 풀이 과정까지 같이 줌
# 3. 조건 나열: "질문의 조건을 모두 나열한 뒤 각각 검토"
# 4. 변형 체크: "이 문제가 기존 유명 문제와 어떻게 다른지 분석하고 답해"


In [ ]:
# CoT 적용 전 (6마리 말 — 답: 1번이어야 함)
result = llm.invoke("말이 6마리가 있고, 가장 빠른 말을 찾고 싶습니다. 한 번에 최대 6마리가 동시에 뛸 수 있습니다. 최소 몇 번의 경주가 필요할까요?")
print(result.content)


In [ ]:
# CoT + 변형 체크 (몬티홀 변형)
# 그냥 물어보면 원본 답(바꿔야 유리)을 뱉기 쉬움
# "단계별로 생각" + "기존 유명 문제와 어떻게 다른지 분석" → 원본과의 차이점 인식 유도
llm.invoke("3개 문 중 하나에 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다. 사회자는 아직 어떤 문도 열지 않았습니다. 바꾸는 것이 유리할까요? 단계별로 생각해봅시다")


In [ ]:
# CoT 적용: 철자 세기
print(llm.invoke("LOLLAPALOOZA 라는 단어에서 L이 몇 번 나타나나요? 단계별로 생각해보세요").content)


In [ ]:
# CoT 적용 + 변형 체크 (몬티홀 변형)
print(llm.invoke("3개 문 중 하나에 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다. 사회자는 아직 어떤 문도 열지 않았습니다. 바꾸는 것이 유리할까요? 단계별로 생각해봅시다, 이 문제가 기존 유명한 문제와 어떻게 다른지를 분석하고 답해줘").content)


In [ ]:
# Sally 문제 + CoT
print(llm.invoke("""Sally(여자) 한테 남자 형제가 3명 있어요
각 남자형제한테는 여자 형제가 2명 있어요
Sally 의 여자 형제는 몇명인가요?
단계별로 생각해보세요""").content)


---
## Part 8. **Few-shot CoT** — 예시에 **풀이까지** 담아서

Zero-shot CoT("단계별로 생각해보세요")로도 부족할 때,
**예시 자체에 [흔한 실수] → [올바른 생각] → [답]** 형태의 풀이를 담아서 보여주는 기법.

### 예시 구조
```
문제: 셔츠 1장 말리는 데 4시간. 셔츠 5장 말리면 몇 시간?
[흔한 실수] 4 × 5 = 20시간
[올바른 생각] 동시에 널면 됩니다.
답: 4시간
```
→ 이렇게 "함정에 빠지지 않는 추론 패턴"을 예시로 보여줌

### 실전 문제
> 친구 집까지 평균 시속 3마일로 걸어갔습니다.
> 왕복 전체 평균 속도를 시속 6마일로 만들려면, 돌아올 때 얼마나 빨리 달려야 할까요?

직관: "6mph면 3mph의 2배니까 12mph로 달리면 되겠지?"
**실제 정답: 불가능** (이미 갈 때 6mph 왕복에 필요한 전체 시간을 다 써버렸음)

### 비유
Few-shot CoT는 **"기출 문제집 해설 보여주기"**. 답만 쓰인 기출이 아니라, **오답 노트 + 풀이 과정**이 같이 들어있는 해설지를 학생(LLM)에게 보여주는 느낌이에요.


In [ ]:
# Few-shot CoT용 시스템 프롬프트 — "흔한 실수 / 올바른 생각" 구조의 예시 포함
system_prompt = """당신은 퍼즐 전문가입니다. 아래 문제들은 유명한 문제의 **변형**입니다.
원래 문제의 답을 그대로 적용하지 마세요. 이 문제의 조건을 정확히 읽고 답하세요.

예시:
문제 : 셔츠 1장을 말리는데 4시간이 걸립니다. 셔츠 5장을 말리면 몇 시간?
[흔한 실수] 4 x 5 = 20시간
[올바른 생각] 동시에 널면 됩니다. 답 : 4시간

이처럼 문제의 조건을 주의 깊게 읽고 단계별로 생각하세요
"""

query = "친구 집까지 평균 시속 3마일로 걸어갔습니다. 왕복 전체 평균 속도를 시속 6마일로 만들려면, 돌아올 때 얼마나 빨리 달려야 할까요?"


In [ ]:
# CoT 미적용 — 직감으로 "돌아올 때 더 빨리 달리면 돼" 식 답변
print(llm.invoke(f"{query}").content)


In [ ]:
# Few-shot CoT 적용 — 시스템 프롬프트의 "흔한 실수" 패턴을 학습하고 조건 재검토
print(llm.invoke(f"{system_prompt}\n{query}").content)
# → 불가능함을 논리적으로 설명해 줄 가능성 ↑


In [ ]:
# 더 강한 CoT: "단계를 명시적으로 지시" (프로시저 CoT)
# 각 단계를 번호로 강제하면 모델이 빼먹기 힘듦
prompt = """
당신은 물리 문제를 푸는 전문가입니다. 절대 직감이나 추측으로 답하지 말고, 정의에 따라 단계적으로 계산하세요.

문제 :
한 사람이 친구 집까지 평균 시속 3마일로 걸어갔습니다. 왕복 전체 평균 속도를 시속 6마일로 만들려면, 돌아올 때 얼마나 빨리 달려야 할까요?

다음 순서를 반드시 따르세요

1. 편도 거리를 D마일이라고 두세요
2. 갈 때 걸린 시간을 계산하세요
3. 왕복 전체 평균 속도가 6mph가 되기 위한 총 시간을 계산하세요
4. 이미 사용한 시간과 필요한 총 시간을 비교하세요
5. 남은 시간으로 돌아오는 것이 가능한지 판단하세요
6. 가능하다면 필요한 속도를 계산하고, 불가능하면 왜 불가능한지 논리적으로 설명하세요
"""
print(llm.invoke(prompt).content)
# → 거의 확실하게 "불가능" 결론 도출


---
## 오늘의 정리

### Few-shot 예시 선택 3원칙
- **다양성**: 여러 케이스 커버
- **관련성**: 태스크와 같은 도메인
- **균형**: 클래스 분포 고르게

### 기술 스택
- `FewShotPromptTemplate` (문자열) / `FewShotChatMessagePromptTemplate` (채팅)
- `SemanticSimilarityExampleSelector` + `InMemoryVectorStore` → 동적 예시 선택
- `pd.DataFrame`으로 평가 결과 시각화

### LLM이 약한 영역 & 해결
| 약점 | CoT 기법 |
|------|----------|
| 한 번에 답 안 나오는 추론 | **"단계별로 생각해 보세요"** (Zero-shot CoT) |
| 유명 문제 패턴 복사 | **"이 문제가 기존과 어떻게 다른지 분석"** |
| 직관 오류 (물리/확률) | **Few-shot CoT** (흔한 실수 + 올바른 생각 예시) |
| 절차 누락 | **단계 번호 매긴 프로시저 CoT** |

### 기억할 한 문장
> **프롬프트 한 줄("단계별로 생각해 보세요")이 모델 성능을 바꾼다.**
> 모델을 바꾸기 전에 프롬프트부터 바꿔 보자.

### 다음 시간 예고 (Day 4)
**Memory** — LLM은 stateless. 대화 맥락을 유지하는 법:
- Buffer / Window / Summarize 메모리
- `RunnableWithMessageHistory`로 체인에 메모리 붙이기
